# 01 — Idosos morando sozinhos (Censo 2022 / IBGE, tabela 9879)

**Objetivo desta etapa:** obter, para os 645 municípios de São Paulo, quantos
domicílios têm responsável idoso (60+) e quantos desses são domicílios
**unipessoais** (uma só pessoa — ou seja, o idoso mora sozinho).

**Status: já temos dado real, testado e validado.** Você conseguiu a
**Tabela 9879 do SIDRA** — "Domicílios particulares, por espécie de unidade
doméstica, número de moradores, segundo sexo, cor ou raça e grupos de idade
da pessoa responsável pelo domicílio", filtrada para SP, Censo 2022. Essa é
a tabela certa (as tentativas anteriores, tabela 9605, traziam população
por cor/raça em "Concentração Urbana" — outra coisa. Ver
`data/external/FONTES_RIO_CLARO.md` para o histórico).

**O que a tabela mede, exatamente:**
- `domicilios_resp_idoso` = nº de domicílios cujo responsável tem 60+ anos
  (proxy do nº de idosos "chefes de domicílio" — não é o total de idosos do
  município, que inclui também idosos que moram na casa de outra pessoa)
- `idosos_sozinhos` = desses, quantos são domicílios **unipessoais** (por
  definição, 1 domicílio unipessoal = 1 morador = o próprio idoso responsável
  morando sozinho) — esse número é exato, não é proxy.

**Validação feita:** a soma de `domicilios_resp_idoso` e de `idosos_sozinhos`
em todos os 645 municípios bate exatamente com os totais do estado de SP
impressos no próprio arquivo (4.592.203 e 1.336.761).

**Saída desta etapa:** `data/processed/censo_domicilios_sp.csv`, uma linha
por município, com as colunas acima mais `pct_idosos_sozinhos`.


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..") / "src"))
import config  # noqa: E402

import pandas as pd


## 1.1 Parser da tabela 9879 (SIDRA)

O CSV do SIDRA vem em UTF-8, `;`-separado, com 7 colunas e cabeçalho em
múltiplas linhas hierárquicas (efeito de exportar uma tabela com vários
cruzamentos). As 3 últimas colunas são sempre iguais entre si (são o mesmo
recorte "Unipessoal" repetido) — só usamos a primeira delas.


In [ ]:
caminho = config.DATA_EXTERNAL / "tabela9879_domicilios_sp.csv"

with open(caminho, encoding="utf-8") as f:
    linhas = f.readlines()

# as linhas de dado de verdade começam em '"Brasil";"Total";"Total";...'
# e terminam antes da linha "Fonte: ..."
inicio = next(i for i, l in enumerate(linhas) if l.startswith('"Brasil";"Total";"Total"'))
fim = next(i for i, l in enumerate(linhas) if l.startswith('"Fonte:'))
miolo = "".join(linhas[inicio:fim])

colunas = ["local", "cor_raca", "faixa_idade", "total_domicilios", "unipessoal", "_u2", "_u3"]
bruto = pd.read_csv(pd.io.common.StringIO(miolo), sep=";", quotechar='"', header=None, names=colunas)

# fica só com as linhas de município (têm o sufixo "(SP)"; descarta Brasil e São Paulo/UF)
municipios = bruto[bruto["local"].str.endswith("(SP)")].copy()
municipios["municipio"] = municipios["local"].str.replace(r"\s*\(SP\)\s*$", "", regex=True)
municipios["municipio_norm"] = municipios["local"].apply(config.normalizar_municipio)

print(f"{municipios['municipio'].nunique()} municípios encontrados (esperado: 645)")
municipios.head()


## 1.2 Separar total x responsável idoso, e salvar


In [ ]:
total = municipios[municipios["faixa_idade"] == "Total"][
    ["municipio", "municipio_norm", "total_domicilios", "unipessoal"]
].rename(columns={"total_domicilios": "domicilios_total", "unipessoal": "domicilios_unipessoais_total"})

idosos = municipios[municipios["faixa_idade"] == "60 anos ou mais"][
    ["municipio_norm", "total_domicilios", "unipessoal"]
].rename(columns={"total_domicilios": "domicilios_resp_idoso", "unipessoal": "idosos_sozinhos"})

censo = total.merge(idosos, on="municipio_norm", how="inner")
censo["pct_idosos_sozinhos"] = (censo["idosos_sozinhos"] / censo["domicilios_resp_idoso"] * 100).round(1)

print(censo.shape)

# validação: confere com os totais do estado de SP impressos no próprio arquivo
print("domicilios_total:", censo["domicilios_total"].sum(), "(esperado 16241500)")
print("domicilios_resp_idoso:", censo["domicilios_resp_idoso"].sum(), "(esperado 4592203)")
print("idosos_sozinhos:", censo["idosos_sozinhos"].sum(), "(esperado 1336761)")

censo.head()


In [ ]:
rio_claro = censo[censo["municipio"] == config.RIO_CLARO_NOME]
print("Rio Claro:")
rio_claro


In [ ]:
censo.to_csv(config.DATA_PROCESSED / "censo_domicilios_sp.csv", index=False)
print("Salvo em", config.DATA_PROCESSED / "censo_domicilios_sp.csv")


## 1.3 Nota metodológica

`domicilios_resp_idoso` não é exatamente "número de idosos do município" —
é o número de **domicílios cujo responsável é idoso**. Um idoso que mora na
casa de um filho, por exemplo, não é contado aqui (o responsável do
domicílio pode ser o filho). Por isso `pct_idosos_sozinhos` deve ser lido
como "% dos domicílios chefiados por idoso que são unipessoais", não como
"% dos idosos do município que moram sozinhos" — a diferença é sutil mas
importante para a seção de Métodos do artigo.

Isso já apareceu de forma concreta em Rio Claro: o ofício da Prefeitura
(notebook 05) registra 37.038 *pessoas* idosas em 2022, contra 21.997
*domicílios* com responsável idoso aqui — a diferença é justamente os
idosos que moram acompanhados sem serem o responsável pelo domicílio.
